# **Epigenetics: *GPNMB*+ vs Homeostatic cells**

This notebook compares the epigenetic profiles of GPNMB+ and homeostatic microglial nuclei across two single-nucleus Multiome datasets (ATAC-seq) from the sEOAD vs Control study.

**Key design decisions:**
- **Dataset 8**: 21 samples (GSMs) are mapped to **9 biological donors** (each donor contributes 1–3 brain regions). Donor grouping is used as the random effect to avoid pseudoreplication.
- **Dataset 9**: Individual samples correspond to unique donors.
- The LMM tests whether `is_activated` (GPNMB+ vs Homeostatic cluster membership) predicts `n_peaks` (global chromatin accessibility), with donor as a random intercept.

In [ ]:
# ============================================================================
# Import libraries
# ============================================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle

from scipy.stats import combine_pvalues
from scipy.stats import ttest_rel
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

In [ ]:
# ============================================================================
# Load single-nucleus Multiome datasets (paired RNA + ATAC)
# ============================================================================

ds8 = sc.read_h5ad('Dataset_8.h5ad')
ds9 = sc.read_h5ad('Dataset_9.h5ad')
ds8_atac = sc.read_h5ad('Dataset_8_ATAC.h5ad')
ds9_atac = sc.read_h5ad('Dataset_9_ATAC.h5ad')

In [ ]:
# Read the LINGER output file (for Dataset 8 only)
df_linger_data = pd.read_csv("LINGER_cis.txt", sep="\t")

# Read the reference GTF annotation file directly from the dataset folder
df_gtf = pd.read_csv(
    "genomic.gtf", 
    sep="\t", 
    comment="#", 
    names=["chrom", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"]
)

# Extract gene_name from the 'attribute' column for clean mapping inside the function
# This ensures df_gtf['gene_name'] column exists and filters properly
df_gtf["gene_name"] = df_gtf["attribute"].str.extract(r'gene_id "([^"]+)"')

In [ ]:
# ============================================================================
# Donor mapping for Dataset 8 (from GSE272082 series matrix)
# ============================================================================
# The .h5ad file stores GSM IDs in 'sample' but has no donor column.
# We map each GSM to its biological donor.
# Key correction: UT04 and UT09 are Control (not sEOAD as mislabeled in the raw h5ad).

gsm_to_donor = {
    "GSM8392674": "Donor1", "GSM8392675": "Donor1", "GSM8392676": "Donor1",
    "GSM8392677": "Donor2", "GSM8392678": "Donor2", "GSM8392679": "Donor2",
    "GSM8392680": "Donor3", "GSM8392681": "Donor3", "GSM8392682": "Donor3",
    "GSM8392683": "Donor4", "GSM8392684": "Donor4", "GSM8392685": "Donor4",
    "GSM8392686": "Donor5", "GSM8392687": "Donor5", "GSM8392688": "Donor5",
    "GSM8392689": "Donor6", "GSM8392690": "Donor6", "GSM9294031": "Donor6",
    "GSM8392691": "Donor7",   # UT04: Control (corrected from sEOAD)
    "GSM8392692": "Donor8",   # UT09: Control (corrected from sEOAD)
    "GSM8392693": "Donor9",   # UT2105: sEOAD
}

# Apply donor mapping to both RNA and ATAC objects for Dataset 8
ds8.obs["donor"] = ds8.obs["sample"].map(gsm_to_donor)
ds8_atac.obs["donor"] = ds8_atac.obs["sample"].map(gsm_to_donor)

print("Dataset 8 donor mapping applied:")
print(f"  Donors: {ds8.obs['donor'].nunique()} unique")
print(f"  Cells per donor:")
print(ds8.obs.groupby("donor").size())

## **Myeloid lineage and activation marker expression**

In [ ]:
# ============================================================================
# Custom dotplot function for gene expression visualization
# ============================================================================

def plot_custom_dotplot(adata, gene_list, groupby='leiden_res_2.0', title="Custom Gene Set"):
    """
    Generates a standardized dotplot for a custom list of genes.
    Forces gene symbols to be italicized for publication standard.
    """
    adata_plot = adata.copy()
    adata_plot.var_names = adata_plot.var_names.astype(str)
    adata_plot = adata_plot[:, ~adata_plot.var_names.duplicated(keep='first')].copy()

    genes = [g for g in gene_list if g in adata_plot.var_names]
    if not genes:
        print(f"Warning: None of the target genes were found in the dataset.")
        return

    dp = sc.pl.dotplot(
        adata_plot,
        var_names=genes,
        groupby=groupby,
        use_raw=False,
        standard_scale='var',
        title=title,
        figsize=(12, 4),
        dendrogram=False,
        cmap='viridis',
        show=False
    )

    if isinstance(dp, dict):
        ax = dp['mainplot_ax']
    else:
        ax = dp.get_axes()['mainplot_ax']

    labels = [label.get_text() for label in ax.get_xticklabels()]
    ax.set_xticklabels(labels, fontdict={'style': 'italic'})

    plt.show()

# Target gene panel for myeloid lineage profiling
target_genes = [
    # Macrophage markers
    "F13A1", "CD163", "CD163L1", "COLEC12",
    # Uncategorized transition genes
    "GRID2", "CCDC26",
    # Homeostatic
    "P2RY12", "CX3CR1", "SORL1", "MEF2A", "ITPR2", "FRMD4A", "ELMO1", "ANKRD44", "SRGAP2",
    # GPNMB+ signature
    "GPNMB", "MITF", "PPARG", "PTPRG", "MYO1E", "CPM", "KCNMA1", "ATG7", "IQGAP2",
    "STARD13", "DPYD", "LRRK2", "FOXP1", "APOE", "SERPINE1",
    # Inflammation & Related
    "SPP1", "TMEM163", "MSR1", "SLC11A1", "CD83", "IL1B", "IRAK2",
    # Ribosomal & Immune
    "RPL32", "RPS19", "C1QB", "FTH1", "TMSB10", "HLA-B",
    # HSPs
    "HSPH1", "DNAJB1", "HSP90AA1"
]

In [ ]:
plot_custom_dotplot(ds8, target_genes, title="Dataset 8: Myeloid Lineage & Activation Markers")

In [ ]:
plot_custom_dotplot(ds9, target_genes, title="Dataset 9: Myeloid Lineage & Activation Markers")

## **Comparing global chromatin accessibility (n_peaks)**

We evaluated the distribution of the number of accessible peaks per nucleus within each pre-selected cluster group:
- **Dataset 8**: Cluster 2 (GPNMB+) vs Clusters 0+1 (Homeostatic)
- **Dataset 9**: Clusters 6+8 (GPNMB+) vs Clusters 2+4+5 (Homeostatic)

Donor-level paired visualizations are overlaid on the cell-level violin plots.

In [ ]:
# ============================================================================
# Prepare cell-level and donor-level (paired) DataFrames
# ============================================================================

# --- Dataset 8: cell-level and donor-paired tables ---

# Compute n_peaks per cell
n_peaks_vector_8 = np.asarray(ds8_atac.X.sum(axis=1)).flatten()
ds8_atac.obs["n_peaks"] = n_peaks_vector_8

# Filter to relevant clusters (0, 1 = Homeostatic; 2 = GPNMB+)
d8_obs = ds8_atac.obs[ds8_atac.obs["leiden_res_2.0"].isin(["0", "1", "2"])].copy()
d8_obs["donor"] = d8_obs["sample"].map(gsm_to_donor)
d8_obs["cell_state"] = d8_obs["leiden_res_2.0"].apply(
    lambda x: "Activated (GPNMB+)" if x == "2" else "Homeostatic"
)

# Cell-level DataFrame (for background violins)
df8_cells = pd.DataFrame({
    "n_peaks": d8_obs["n_peaks"].values,
    "state": d8_obs["cell_state"].values
})

# Donor-level paired table (for overlay dots and connecting lines)
d8_pivot = d8_obs.groupby(["donor", "cell_state"])["n_peaks"].mean().unstack()
d8_pivot = d8_pivot.dropna().reset_index()  # keep only donors with both states

np.random.seed(42)
colors_d8 = sns.color_palette("hls", len(d8_pivot))

df8_paired = pd.DataFrame({
    "donor": d8_pivot["donor"],
    "Homeostatic": d8_pivot["Homeostatic"],
    "Activated": d8_pivot["Activated (GPNMB+)"],
    "jitter": np.random.uniform(-0.15, 0.15, size=len(d8_pivot)),
    "color": [colors_d8[i] for i in range(len(d8_pivot))]
})

# --- Dataset 9: cell-level and donor-paired tables ---

n_peaks_vector_9 = np.asarray(ds9_atac.X.sum(axis=1)).flatten()
ds9_atac.obs["n_peaks"] = n_peaks_vector_9

# Filter to relevant clusters (2, 4, 5 = Homeostatic; 6, 8 = GPNMB+)
d9_obs = ds9_atac.obs[ds9_atac.obs["leiden_res_2.0"].isin(["2", "4", "5", "6", "8"])].copy()
d9_obs["cell_state"] = d9_obs["leiden_res_2.0"].apply(
    lambda x: "Activated (GPNMB+)" if x in ["6", "8"] else "Homeostatic"
)

# Cell-level DataFrame
df9_cells = pd.DataFrame({
    "n_peaks": d9_obs["n_peaks"].values,
    "state": d9_obs["cell_state"].values
})

# Donor-level paired table
d9_pivot = d9_obs.groupby(["sample", "cell_state"])["n_peaks"].mean().unstack()
d9_pivot = d9_pivot.dropna().reset_index()

colors_d9 = sns.color_palette("hls", len(d9_pivot))

df9_paired = pd.DataFrame({
    "donor": d9_pivot["sample"],
    "Homeostatic": d9_pivot["Homeostatic"],
    "Activated": d9_pivot["Activated (GPNMB+)"],
    "jitter": np.random.uniform(-0.2, 0.2, size=len(d9_pivot)),
    "color": [colors_d9[i] for i in range(len(d9_pivot))]
})

print(f"Dataset 8: {len(df8_paired)} paired donors (out of 9 total)")
print(f"Dataset 9: {len(df9_paired)} paired donors")

In [ ]:
# ============================================================================
# Paired violin + donor-line visualization
# ============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6), sharey=False, dpi=300)

state_order = ["Homeostatic", "Activated (GPNMB+)"]
bg_palette = {"Activated (GPNMB+)": "#ff9287", "Homeostatic": "#8bd1ff"}

# --- Panel 1: Dataset 8 ---
sns.violinplot(
    data=df8_cells, x="state", y="n_peaks", order=state_order, hue="state",
    palette=bg_palette, dodge=False, inner="quartile", linewidth=1.5, edgecolor="#656565", ax=ax1
)
ax1.collections[0].set_alpha(0.6)
ax1.collections[1].set_alpha(0.6)

for _, row in df8_paired.iterrows():
    j = row["jitter"]
    x_homeo = 0 + j
    x_active = 1 + j
    y_homeo = row["Homeostatic"]
    y_active = row["Activated"]
    c = row["color"]

    ax1.plot([x_homeo, x_active], [y_homeo, y_active], color=c, alpha=1, linestyle="-", linewidth=1.5, zorder=3)
    ax1.scatter([x_homeo, x_active], [y_homeo, y_active], color=c, edgecolors="black", linewidths=0.5, alpha=0.9, s=45, zorder=4)

ax1.set_ylim(-1000, 300000)
ax1.set_xlabel("")
ax1.set_ylabel("Number of Accessible Peaks", fontsize=14)
ax1.set_xticklabels(state_order, fontsize=12)
if ax1.get_legend(): ax1.get_legend().remove()


# --- Panel 2: Dataset 9 ---
sns.violinplot(
    data=df9_cells, x="state", y="n_peaks", order=state_order, hue="state",
    palette=bg_palette, dodge=False, inner="quartile", linewidth=1.5, edgecolor="#656565", ax=ax2
)
ax2.collections[0].set_alpha(0.6)
ax2.collections[1].set_alpha(0.6)

for _, row in df9_paired.iterrows():
    j = row["jitter"]
    x_homeo = 0 + j
    x_active = 1 + j
    y_homeo = row["Homeostatic"]
    y_active = row["Activated"]
    c = row["color"]

    ax2.plot([x_homeo, x_active], [y_homeo, y_active], color=c, alpha=1, linestyle="-", linewidth=1.2, zorder=3)
    ax2.scatter([x_homeo, x_active], [y_homeo, y_active], color=c, edgecolors="black", linewidths=0.5, alpha=0.9, s=35, zorder=4)

ax2.set_ylim(-5000, 300000)
ax2.set_xlabel("")
ax2.set_ylabel("")
ax2.set_xticklabels(state_order, fontsize=12)
if ax2.get_legend(): ax2.get_legend().remove()

fig.suptitle("Global Chromatin Accessibility Shifts across Paired Donor Cohorts",
             fontsize=18, fontweight="bold", y=1.02)

sns.despine(fig=fig)
plt.tight_layout()
plt.show()

## **Linear Mixed-Effects Model (LMM)**

To test whether GPNMB+ cells have significantly higher chromatin accessibility than homeostatic cells, we fit a linear mixed-effects model:

$$\text{n\_peaks} \sim \text{is\_activated} + (1|\text{donor})$$

where `is_activated = 1` for GPNMB+ clusters and `0` for homeostatic clusters. The random intercept for donor accounts for within-donor correlation and prevents pseudoreplication.

**Dataset 8**: groups by `donor` (9 unique donors)  
**Dataset 9**: groups by `sample` (individual donors)

In [ ]:
# ============================================================================
# Linear Mixed-Effects Model: n_peaks ~ is_activated + (1|donor)
# ============================================================================

# --- Dataset 8 ---
print("Processing Linear Mixed-Effects Model for Dataset 8...")

n_peaks_vector_8 = np.asarray(ds8_atac.X.sum(axis=1)).flatten()
peaks_dict_8 = dict(zip(ds8_atac.obs_names, n_peaks_vector_8))
d8_filtered = ds8_atac.obs[ds8_atac.obs['leiden_res_2.0'].isin(['0', '1', '2'])].copy()

df_lmm_8 = pd.DataFrame({
    'n_peaks': d8_filtered.index.map(peaks_dict_8).astype(float),
    'is_activated': d8_filtered['leiden_res_2.0'].apply(lambda x: 1 if x == '2' else 0),
    'donor_id': d8_filtered['donor'].astype(str)
})

model_8 = smf.mixedlm("n_peaks ~ is_activated", data=df_lmm_8, groups=df_lmm_8["donor_id"])
mdf_8 = model_8.fit()

p_value_8 = mdf_8.pvalues['is_activated']
coef_8 = mdf_8.params['is_activated']
stderr_8 = mdf_8.bse['is_activated']

# --- Dataset 9 ---
print("Processing Linear Mixed-Effects Model for Dataset 9...")

n_peaks_vector_9 = np.asarray(ds9_atac.X.sum(axis=1)).flatten()
peaks_dict_9 = dict(zip(ds9_atac.obs_names, n_peaks_vector_9))
d9_filtered = ds9_atac.obs[ds9_atac.obs['leiden_res_2.0'].isin(['2', '4', '5', '6', '8'])].copy()

df_lmm_9 = pd.DataFrame({
    'n_peaks': d9_filtered.index.map(peaks_dict_9).astype(float),
    'is_activated': d9_filtered['leiden_res_2.0'].apply(lambda x: 1 if x in ['6', '8'] else 0),
    'sample_id': d9_filtered['sample'].astype(str)
})

model_9 = smf.mixedlm("n_peaks ~ is_activated", data=df_lmm_9, groups=df_lmm_9["sample_id"])
mdf_9 = model_9.fit()

p_value_9 = mdf_9.pvalues['is_activated']
coef_9 = mdf_9.params['is_activated']
stderr_9 = mdf_9.bse['is_activated']

# --- Meta-Analysis: combine p-values across cohorts ---
print("Performing Meta-Analysis across cohorts...")

empirical_p_values = [p_value_8, p_value_9]

# Method A: Unweighted Fisher's method
_, p_fisher = combine_pvalues(empirical_p_values, method='fisher')

# Method B: Sample size-weighted Stouffer's method
cell_weights = [len(df_lmm_8), len(df_lmm_9)]
_, p_stouffer = combine_pvalues(empirical_p_values, method='stouffer', weights=cell_weights)

# --- Report ---
print("\n" + "="*70)
print("             MANUSCRIPT META-ANALYSIS SUMMARY REPORT")
print("="*70)
print(f"Dataset 8 LMM Fixed Effect (Coef): {coef_8:.4f} (SE: {stderr_8:.4f})")
print(f"Dataset 8 Empirical P-value:      {p_value_8:.4e}")
print(f"Dataset 8 Analytical Cell Count:   {cell_weights[0]}")
print("-"*70)
print(f"Dataset 9 LMM Fixed Effect (Coef): {coef_9:.4f} (SE: {stderr_9:.4f})")
print(f"Dataset 9 Empirical P-value:      {p_value_9:.4e}")
print(f"Dataset 9 Analytical Cell Count:   {cell_weights[1]}")
print("="*70)
print(f"Combined P-value (Fisher's Unweighted):   {p_fisher:.4e}")
print(f"Combined P-value (Weighted Stouffer's):   {p_stouffer:.4e}")
print("="*70)

## **Differential Accessibility Regions (DARs)**

We identified differentially accessible regions (DARs) between GPNMB+ and homeostatic nuclei using the Wilcoxon rank-sum test (equivalent to marker gene detection in snRNA-seq).

- **Dataset 8**: Cluster 2 (GPNMB+) vs Clusters 0+1 (Homeostatic)
- **Dataset 9**: Clusters 6+8 (GPNMB+) vs Clusters 2+4+5 (Homeostatic)

The following code was used to compute DARs (computation takes several hours; results are pre-saved as CSV and loaded below):

In [ ]:
# ==============================================================================
# DIFFERENTIAL ACCESSIBILITY REGION (DAR) ANALYSIS
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. DATASET 8: Cluster 2 (Activated) vs Clusters 0 + 1 (Homeostasis)
# ------------------------------------------------------------------------------
print("Running DAR analysis for Dataset 8...")

# Create a logical mask to isolate only target clusters
cells_subset_d8 = ds8_atac.obs["leiden_res_2.0"].isin(["0", "1", "2"])
adata_subset_d8 = ds8_atac[cells_subset_d8].copy()

# Collapse Clusters 0 and 1 into a single reference group named 'Homeostasis'
adata_subset_d8.obs["comparison_group"] = adata_subset_d8.obs["leiden_res_2.0"].map({
    "0": "Homeostasis",
    "1": "Homeostasis",
    "2": "Activated"
})

# Execute Differential Accessibility analysis using Wilcoxon rank-sum test
sc.tl.rank_genes_groups(
    adata_subset_d8,
    groupby="comparison_group",
    groups=["Activated"],
    reference="Homeostasis",
    method="wilcoxon",
    use_raw=False
)

# Extract statistics into a clean DataFrame
result_d8 = sc.get.rank_genes_groups_df(adata_subset_d8, group="Activated")
print(f"✅ Dataset 8: Found {len(result_d8[result_d8['pvals_adj'] < 0.05])} significant peaks (FDR < 0.05)")


# ------------------------------------------------------------------------------
# 2. DATASET 9: Clusters 6 + 8 (Activated) vs Clusters 2 + 4 + 5 (Homeostasis)
# ------------------------------------------------------------------------------
print("\nRunning DAR analysis for Dataset 9...")

# Create a logical mask to isolate only target clusters
target_clusters_d9 = ["2", "4", "5", "6", "8"]
cells_subset_d9 = ds9_atac.obs["leiden_res_2.0"].isin(target_clusters_d9)
adata_subset_d9 = ds9_atac[cells_subset_d9].copy()

# Map specific clusters to 'Activated' and 'Homeostasis' groups respectively
adata_subset_d9.obs["comparison_group"] = adata_subset_d9.obs["leiden_res_2.0"].map({
    "2": "Homeostasis",
    "4": "Homeostasis",
    "5": "Homeostasis",
    "6": "Activated",
    "8": "Activated"
})

# Execute Differential Accessibility analysis using Wilcoxon rank-sum test
sc.tl.rank_genes_groups(
    adata_subset_d9,
    groupby="comparison_group",
    groups=["Activated"],
    reference="Homeostasis",
    method="wilcoxon",
    use_raw=False
)

# Extract statistics into a clean DataFrame
result_d9 = sc.get.rank_genes_groups_df(adata_subset_d9, group="Activated")
print(f"✅ Dataset 9: Found {len(result_d9[result_d9['pvals_adj'] < 0.05])} significant peaks (FDR < 0.05)")

# Map peak coordinates to gene annotations from .var dataframe
result_d8["gene_annotation"] = result_d8["names"].map(ds8_atac.var["gene_ann"])
result_d9["gene_annotation"] = result_d9["names"].map(ds9_atac.var["gene_ann"])

# Save the dataframes
result_d8.to_csv("Dataset_8_DARs_Results.csv", index=False)
result_d9.to_csv("Dataset_9_DARs_Results.csv", index=False)

In [ ]:
# ==============================================================================
# Load pre-computed DAR results
# ==============================================================================
result_d8 = pd.read_csv("Dataset_8_DARs_Results.csv")
result_d9 = pd.read_csv("Dataset_9_DARs_Results.csv")

n_sig_d8 = (result_d8["pvals_adj"] < 0.05).sum()
n_sig_d9 = (result_d9["pvals_adj"] < 0.05).sum()
print(f"Dataset 8: {len(result_d8):,} total regions tested, {n_sig_d8:,} significant DARs (FDR < 0.05)")
print(f"Dataset 9: {len(result_d9):,} total regions tested, {n_sig_d9:,} significant DARs (FDR < 0.05)")

In [ ]:
# ==============================================================================
# Volcano plots for DARs
# ==============================================================================

def plot_single_volcano(df, ax, title, selection_criterion, p_threshold=0.05, lfc_threshold=0.5):
    """
    Generates a stylized volcano plot with top differentially accessible regions labeled.
    """
    df = df.copy()
    df['log_p'] = -np.log10(df['pvals_adj'] + 1e-300)
    
    df['significance'] = 'Non-significant'
    df.loc[(df['pvals_adj'] < p_threshold) & (df['logfoldchanges'] > lfc_threshold), 'significance'] = 'Upregulated'
    df.loc[(df['pvals_adj'] < p_threshold) & (df['logfoldchanges'] < -lfc_threshold), 'significance'] = 'Downregulated'
    
    color_map = {'Non-significant': '#bdc3c7', 'Upregulated': '#e74c3c', 'Downregulated': '#3498db'}
    sns.scatterplot(
        data=df, x='logfoldchanges', y='log_p', hue='significance',
        palette=color_map, alpha=0.5, s=12, edgecolor=None, ax=ax
    )
    
    ax.axhline(-np.log10(p_threshold), color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(lfc_threshold, color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(-lfc_threshold, color='black', linestyle='--', linewidth=1, alpha=0.7)
    
    labeled_pool = df[(df['gene_annotation'].notna()) & (df['gene_annotation'] != 'Intergenic')]
    
    if selection_criterion == 'p_value':
        top_up = labeled_pool[labeled_pool['significance'] == 'Upregulated'].nlargest(10, 'log_p')
        top_down = labeled_pool[labeled_pool['significance'] == 'Downregulated'].nlargest(10, 'log_p')
        sub_title = f"{title}\n[Top 10 labeled by Significance (p-value)]"
    elif selection_criterion == 'log_fc':
        top_up = labeled_pool[labeled_pool['significance'] == 'Upregulated'].nlargest(10, 'logfoldchanges')
        top_down = labeled_pool[labeled_pool['significance'] == 'Downregulated'].nsmallest(10, 'logfoldchanges')
        sub_title = f"{title}\n[Top 10 labeled by Magnitude (Log2FC)]"
        
    up_genes_list = "\n".join([f"\u2022 {g}" for g in top_up['gene_annotation'].unique() if pd.notna(g)])
    down_genes_list = "\n".join([f"\u2022 {g}" for g in top_down['gene_annotation'].unique() if pd.notna(g)])
    
    props = dict(boxstyle='round,pad=0.5', facecolor='#ffffff', edgecolor='#dcdde1', alpha=0.95, lw=1)
    
    if up_genes_list:
        ax.text(
            0.95, 0.92, f"Top Upregulated:\n{up_genes_list}",
            transform=ax.transAxes, fontsize=8.5, color='#c0392b', fontweight='bold',
            verticalalignment='top', horizontalalignment='right', bbox=props
        )
    if down_genes_list:
        ax.text(
            0.05, 0.92, f"Top Downregulated:\n{down_genes_list}",
            transform=ax.transAxes, fontsize=8.5, color='#2980b9', fontweight='bold',
            verticalalignment='top', horizontalalignment='left', bbox=props
        )
        
    ax.set_title(sub_title, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Log2 Fold Change', fontsize=12)
    ax.set_ylabel('-Log10 Adjusted P-value', fontsize=12)
    ax.tick_params(labelsize=11)
    if ax.get_legend(): ax.get_legend().remove()

# ==============================================================================
# Build 2x2 volcano plot grid
# ==============================================================================
fig, axs = plt.subplots(2, 2, figsize=(18, 14), dpi=300)

plot_single_volcano(result_d8, axs[0, 0], title="Dataset 8", selection_criterion='p_value')
plot_single_volcano(result_d8, axs[0, 1], title="Dataset 8", selection_criterion='log_fc')
plot_single_volcano(result_d9, axs[1, 0], title="Dataset 9", selection_criterion='p_value')
plot_single_volcano(result_d9, axs[1, 1], title="Dataset 9", selection_criterion='log_fc')

handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=3, fontsize=12, frameon=False)

sns.despine(fig=fig)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

As inferred from the preliminary analysis, GPNMB+ microglia indeed show higher accessibility at GPNMB+ signature genes (though not within the top 10). Additionally, S100 genes are among the top 10 GPNMB+-enriched loci in Dataset 8, which is interesting given their established role in aging and inflammation. Conversely, in homeostatic microglia, we found almost no DARs, with the notable exception of MEIS1, an important repressor according to our TF-target analysis. This aligns with our previous finding that the homeostatic state is generally characterized by closed chromatin.

In [ ]:
# Define a function to illustrate a region with features

def plot_region_combined_with_donors(
    df_gtf,
    ds8_atac,
    ds9_atac,
    result8,
    result9,
    chrom,
    start,
    end,
    df_linger=None,
    genes=None,
    fdr_threshold=0.05,
    figsize=(16, 12),
):
    offset, width = start, end - start

    # --- STYLE CONFIGURATION & PALETTE ---
    C_RED_EXON, C_RED_ARROW = "#b294df", "#7733c2"
    C_BLU_EXON, C_BLU_ARROW = "#b294df", "#7733c2"
    _C_GPNMB, _C_HOMEO, _C_NS = "#ff9287", "#8bd1ff", "#aaaaaa"
    FS_TITLE, FS_LABELS, FS_GENES, FS_SMALL = 23, 23, 18, 18

    # 1. Genomic Features (GTF Processing)
    region_gtf = df_gtf[
        (df_gtf["chrom"] == chrom)
        & (df_gtf["start"] <= end)
        & (df_gtf["end"] >= start)
    ].copy()
    if genes:
        region_gtf = region_gtf[region_gtf["gene_name"].isin(genes)]

    gene_list = sorted(
        region_gtf[region_gtf["feature"] == "gene"]["gene_name"].unique(),
        key=lambda g: region_gtf[
            (region_gtf["gene_name"] == g) & (region_gtf["feature"] == "gene")
        ]["start"].min(),
    )

    # 2. Chromatin Signal Extraction
    var_names = ds8_atac.var_names.to_series()
    p_coords = var_names.str.extract(r"^(.+):(\d+)-(\d+)$").rename(
        columns={0: "chr", 1: "s", 2: "e"}
    )
    p_coords[["s", "e"]] = p_coords[["s", "e"]].astype(int)

    region_peaks = var_names[
        (p_coords["chr"] == chrom)
        & (p_coords["s"] <= end)
        & (p_coords["e"] >= start)
    ].tolist()

    # Dynamic parsing of LINGER cis-regulatory elements from file database
    linger_cis_peaks = set()
    if df_linger is not None:
        # Assuming df_linger has 'Peak' column or coordinates matching the region
        # Adjust column name mapping if your LINGER file uses a different header (e.g., 'peak_id')
        linger_col = "Peak" if "Peak" in df_linger.columns else df_linger.columns[0]
        linger_cis_peaks = set(
            df_linger[df_linger[linger_col].isin(region_peaks)][
                linger_col
            ].tolist()
        )

    # Calculate absolute cell cohort sizes for proper weighted averaging
    n8_g = (ds8_atac.obs["leiden_res_2.0"] == "2").sum()
    n9_g = (ds9_atac.obs["leiden_res_2.0"].isin(["6", "8"])).sum()
    n8_h = (ds8_atac.obs["leiden_res_2.0"].isin(["0", "1"])).sum()
    n9_h = (ds9_atac.obs["leiden_res_2.0"].isin(["2", "4", "5"])).sum()

    # Extract single-cell signals per dataset cohort
    sig8_g = np.asarray(
        ds8_atac[ds8_atac.obs["leiden_res_2.0"] == "2", region_peaks].X.mean(
            axis=0
        )
    ).flatten()
    sig9_g = np.asarray(
        ds9_atac[
            ds9_atac.obs["leiden_res_2.0"].isin(["6", "8"]), region_peaks
        ].X.mean(axis=0)
    ).flatten()
    sig8_h = np.asarray(
        ds8_atac[
            ds8_atac.obs["leiden_res_2.0"].isin(["0", "1"]), region_peaks
        ].X.mean(axis=0)
    ).flatten()
    sig9_h = np.asarray(
        ds9_atac[
            ds9_atac.obs["leiden_res_2.0"].isin(["2", "4", "5"]), region_peaks
        ].X.mean(axis=0)
    ).flatten()

    # Compute global cross-cohort weighted signal tracks
    sig_gpnmb = (sig8_g * n8_g + sig9_g * n9_g) / (n8_g + n9_g)
    sig_homeo = (sig8_h * n8_h + sig9_h * n9_h) / (n8_h + n9_h)

    # 3. Process Peak Differentials and Signatures
    res8 = result8[result8["names"].isin(region_peaks)].set_index("names")
    res9 = result9[result9["names"].isin(region_peaks)].set_index("names")

    def _get_peak_color(p):
        if p not in res8.index or p not in res9.index:
            return _C_NS
        r8, r9 = res8.loc[p], res9.loc[p]
        if (
            r8["pvals_adj"] < fdr_threshold
            and r9["pvals_adj"] < fdr_threshold
        ):
            return _C_GPNMB if r8["logfoldchanges"] > 0 else _C_HOMEO
        return _C_NS

    peak_colors = {p: _get_peak_color(p) for p in region_peaks}

    # 4. Axes Multi-Panel Canvas Setup
    fig, axes = plt.subplots(
        3, 1, figsize=figsize, sharex=True, gridspec_kw={"height_ratios": [2, 3, 3]}
    )
    ax_g, ax_gpnmb, ax_homeo = axes
    y_max = max(sig_gpnmb.max(), sig_homeo.max()) * 1.25
    ax_g.set_title(
        f"{chrom}:{start:,}—{end:,}",
        fontsize=FS_TITLE,
        fontweight="bold",
        pad=25,
    )

    # 5. Gene Annotation Track
    ax_g.set_xlim(0, width)
    ax_g.set_ylim(-0.5, max(len(gene_list), 1) * 0.7)
    for i, g_name in enumerate(gene_list):
        bits = region_gtf[region_gtf["gene_name"] == g_name]
        gene_row = bits[bits["feature"] == "gene"].iloc[0]
        strand, y = gene_row["strand"], i * 0.7
        c_exon, c_arrow = (
            (C_RED_EXON, C_RED_ARROW)
            if strand == "-"
            else (C_BLU_EXON, C_BLU_ARROW)
        )
        g_s, g_e = (
            max(0, gene_row["start"] - offset),
            min(width, gene_row["end"] - offset),
        )

        # Baseline gene locus spine
        ax_g.hlines(y, g_s, g_e, colors=c_arrow, lw=2.5, zorder=1)

        # Collapse exons using dynamic union algorithm
        exons = bits[bits["feature"] == "exon"][["start", "end"]].sort_values(
            "start"
        )
        if not exons.empty:
            merged_exons = []
            curr_s, curr_e = exons.iloc[0]["start"], exons.iloc[0]["end"]
            for _, row in exons.iloc[1:].iterrows():
                if row["start"] <= curr_e:
                    curr_e = max(curr_e, row["end"])
                else:
                    merged_exons.append((curr_s, curr_e))
                    curr_s, curr_e = row["start"], row["end"]
            merged_exons.append((curr_s, curr_e))

            # Render exon block geometry
            for s, e in merged_exons:
                es, ee = max(0, s - offset), min(width, e - offset)
                ax_g.add_patch(
                    Rectangle(
                        (es, y - 0.15),
                        ee - es,
                        0.3,
                        facecolor=c_exon,
                        edgecolor=c_exon,
                        linewidth=0.5,
                        zorder=2,
                    )
                )

        # Transcriptional direction indicator arrow
        arrow_x = g_e if strand == "+" else g_s
        ax_g.annotate(
            "",
            xy=(arrow_x, y),
            xytext=(
                arrow_x - (width * 0.03 if strand == "+" else -width * 0.03),
                y,
            ),
            arrowprops=dict(
                arrowstyle="->", color=c_arrow, lw=4, mutation_scale=25
            ),
            zorder=10,
        )
        ax_g.text(
            g_s,
            y + 0.22,
            g_name,
            fontsize=FS_GENES,
            fontweight="bold",
            fontstyle="italic",
            zorder=11,
        )
    ax_g.axis("off")

    # 6. Render Signal Tracks with Dual-Dataset Cohort Estimations
    def draw_track(ax, global_signals, sig_d8, sig_d9, is_gpnmb=False):
        ax.set_xlim(0, width)
        ax.set_ylim(0, y_max)

        for j, p_id in enumerate(region_peaks):
            ps = max(0, p_coords.loc[p_id, "s"] - offset)
            pe = min(width, p_coords.loc[p_id, "e"] - offset)
            midpoint = ps + (pe - ps) / 2

            # Main global peak envelope bar
            ax.add_patch(
                Rectangle(
                    (ps, 0),
                    pe - ps,
                    global_signals[j],
                    facecolor=peak_colors[p_id],
                    alpha=0.7 if peak_colors[p_id] != _C_NS else 0.35,
                )
            )

            # Overlay individual dataset empirical estimations if significant
            if peak_colors[p_id] != _C_NS:
                # Dataset 8 estimation marker: Black Circle
                ax.plot(
                    midpoint,
                    sig_d8[j],
                    marker="o",
                    color="black",
                    markersize=7,
                    alpha=0.8,
                    zorder=15,
                )
                # Dataset 9 estimation marker: Black Triangle
                ax.plot(
                    midpoint,
                    sig_d9[j],
                    marker="^",
                    color="black",
                    markersize=8,
                    alpha=0.8,
                    zorder=15,
                )

            # Hatch pattern for LINGER interactions
            if p_id in linger_cis_peaks:
                ax.add_patch(
                    Rectangle(
                        (ps, 0),
                        pe - ps,
                        global_signals[j],
                        facecolor="none",
                        edgecolor="#222222",
                        hatch="///",
                        alpha=0.3,
                        zorder=4,
                    )
                )

        lbl = "Activated\nClusters" if is_gpnmb else "Homeostatic\nClusters"
        ax.set_ylabel(lbl, fontsize=FS_LABELS, fontweight="bold", labelpad=15)
        ax.tick_params(labelsize=FS_SMALL)

    # Draw both tracks supplying global averages and raw dataset-specific arrays
    draw_track(ax_gpnmb, sig_gpnmb, sig8_g, sig9_g, is_gpnmb=True)
    draw_track(ax_homeo, sig_homeo, sig8_h, sig9_h)

    # Configure genomic coordinate tick intervals across the horizontal span
    ax_homeo.set_xticks(np.linspace(0, width, 5))
    ax_homeo.set_xticklabels(
        [f"{int(t + offset):,}" for t in np.linspace(0, width, 5)], 
        fontsize=FS_SMALL
    )
    ax_homeo.tick_params(axis='x', pad=15)

    # Final aesthetic clean-up and rendering
    sns.despine(fig=fig, top=True, right=True)
    plt.tight_layout()
    plt.show()

In [ ]:
# NC -> CHR chromosome conversion

def convert_refseq_to_chr(chrom_str):
    chrom_str = str(chrom_str)
    # Mitochondial DNA
    if "NC_012920" in chrom_str:
        return "chrM"
    # Usual chromosomes
    if chrom_str.startswith("NC_0000"):
        try:
            num_str = chrom_str[7:9]
            num = int(num_str)
            if num == 23:
                return "chrX"
            elif num == 24:
                return "chrY"
            else:
                return f"chr{num}"
        except ValueError:
            return chrom_str
    return chrom_str

# Apply it to all dataframe
df_gtf["chrom"] = df_gtf["chrom"].apply(convert_refseq_to_chr)

In [ ]:
# S100 region
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr1",  
    start=153510000,  
    end=153575000,  
    df_linger=df_linger_data,  
)

In [ ]:
# IQGAP2
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr5",  
    start=76350000,  
    end=76750000,  
    df_linger=df_linger_data,  
)

In [ ]:
# ZNF804A
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr2",  
    start=184590000,  
    end=184950000,  
    df_linger=df_linger_data,  
)

In [ ]:
# FOXP1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr3",  
    start=70900000,  
    end=71650000,  
    df_linger=df_linger_data,  
)

In [ ]:
# GPNMB
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr7",  
    start=23225000,  
    end=23287500,  
    df_linger=df_linger_data,  
)

In [ ]:
# PTPRG
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr3', 
    start=61500000, 
    end=62250000,
    df_linger=df_linger_data,  
)

In [ ]:
# SLC11A1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr2', 
    start=218375000, 
    end=218410000,
    df_linger=df_linger_data,  
)

In [ ]:
# MYO1E
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr15', 
    start=59100000, 
    end=59400000,
    df_linger=df_linger_data,  
)

In [ ]:
# DPYD
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr1', 
    start=97000000, 
    end=98000000,
    df_linger=df_linger_data,  
)

In [ ]:
# SERPINE1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr7', 
    start=101110000, 
    end=101140000,
    df_linger=df_linger_data,  
)

In [ ]:
# KCNMA1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr10', 
    start=76800000, 
    end=77700000,
    df_linger=df_linger_data,  
)

In [ ]:
# ATG7
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr3', 
    start=11250000, 
    end=11600000,
    df_linger=df_linger_data,  
)

In [ ]:
# MEIS1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom="chr2",  
    start=66400000,  
    end=66600000,  
    df_linger=df_linger_data,  
)

In [ ]:
# SPP1
plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr4', 
    start=87950000, 
    end=88000000,
    df_linger=df_linger_data,  
)

In [ ]:
# PPARG

plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr3', 
    start=12250000, 
    end=12450000,
    df_linger=df_linger_data,  
)

In [ ]:
# MITF

plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr3', 
    start=69700000, 
    end=70000000,
    df_linger=df_linger_data,  
)

In [ ]:
# P2RY12

plot_region_combined_with_donors(
    df_gtf=df_gtf,  
    ds8_atac=ds8_atac,  
    ds9_atac=ds9_atac,  
    result8=result_d8,  
    result9=result_d9,  
    chrom='chr3', 
    start=151200000,
    end=151450000,
    df_linger=df_linger_data,  
)

Thus, while signature genes and the S100A2-6 regions are more open in GPNMB+ clusters, MEIS1 exhibits higher accessibility in homeostatic nuclei.

# **Analyzing the expression of SASP genes**

We decided to additionally evaluate whether genes within the S100A locus also exhibit higher expression in GPNMB+ cells than in homeostatic ones. Additionally, we took SERPINE1, since it also is a component of SASP. To achieve this, we applied a paired pseudobulk analysis, directly comparing GPNMB+ and homeostatic cells within each individual donor.

In [ ]:
# Run pseudobulk for S100A2, S100A3, S100A4, S100A5, S100A6, SERPINE1

def run_rna_pseudobulk(
    adata_obj, 
    dataset_name, 
    cluster_col, 
    act_clusters, 
    homeo_clusters, 
    donor_col, 
    target_genes
):
    """
    Computes pseudobulk expression values profiles per donor/sample 
    and evaluates statistical significance using a paired t-test.
    """
    print(f"Processing RNA Pseudobulk for {dataset_name}...")
    
    # 1. Filter cells to retain only target functional microglial states
    all_target_clusters = act_clusters + homeo_clusters
    cell_mask = adata_obj.obs[cluster_col].isin(all_target_clusters)
    adata_filtered = adata_obj[cell_mask].copy()
    
    # Assign definitive functional state labels
    adata_filtered.obs['functional_state'] = adata_filtered.obs[cluster_col].apply(
        lambda x: 'Activated' if x in act_clusters else 'Homeostasis'
    )
    
    # 2. Match target gene symbols with matrix feature index (Ensembl IDs or Symbols)
    symbol_to_var = dict(zip(adata_filtered.var['gene_symbols'], adata_filtered.var_names))
    valid_genes = [g for g in target_genes if g in symbol_to_var]
    
    if not valid_genes:
        print(f"Warning: None of the target genes found in 'gene_symbols' for {dataset_name}!")
        return pd.DataFrame()
        
    # Map symbols back to internal matrix feature identifiers
    feature_ids = [symbol_to_var[g] for g in valid_genes]
    
    # 3. Extract single-cell expression matrices safely handling sparse inputs
    if hasattr(adata_filtered.X, "toarray"):
        exp_matrix = adata_filtered[:, feature_ids].X.toarray()
    else:
        exp_matrix = np.asarray(adata_filtered[:, feature_ids].X)
        
    # Build core tracking dataframe mapping human-readable symbols to columns
    df_cells = pd.DataFrame(exp_matrix, columns=valid_genes, index=adata_filtered.obs_names)
    df_cells['donor_id'] = adata_filtered.obs[donor_col].astype(str)
    df_cells['state'] = adata_filtered.obs['functional_state']
    
    # 4. Collapse profiles into donor-level means within each state
    df_pseudobulk = df_cells.groupby(['donor_id', 'state'])[valid_genes].mean().reset_index()
    
    # Retain strictly paired biological donors across both microglial states
    donor_counts = df_pseudobulk.groupby('donor_id')['state'].nunique()
    paired_donors = donor_counts[donor_counts == 2].index.tolist()
    df_paired = df_pseudobulk[df_pseudobulk['donor_id'].isin(paired_donors)]
    
    print(f"-> Found {len(paired_donors)} independent biological donors with paired profiles.")
    if df_paired.empty:
        return pd.DataFrame()
        
    # 5. Statistical evaluation via Paired Student's t-test
    results = []
    for gene in valid_genes:
        act_values = df_paired[df_paired['state'] == 'Activated'].sort_values('donor_id')[gene].values
        homeo_values = df_paired[df_paired['state'] == 'Homeostasis'].sort_values('donor_id')[gene].values
        
        mean_act = np.mean(act_values)
        mean_homeo = np.mean(homeo_values)
        
        # Handle mathematical edge cases for division by zero smoothly
        if mean_act == 0 and mean_homeo == 0:
            lfc, p_val = 0.0, 1.0
        else:
            lfc = np.log2((mean_act + 1e-6) / (mean_homeo + 1e-6))
            _, p_val = ttest_rel(act_values, homeo_values)
            
        results.append({
            'Dataset': dataset_name,
            'Gene': gene,
            'Mean_Homeostasis': round(mean_homeo, 4),
            'Mean_Activated': round(mean_act, 4),
            'Log2FC': round(lfc, 3),
            'p_value': p_val
        })
        
    return pd.DataFrame(results)

# ==============================================================================
# EXECUTION: paired pseudobulk with biological donor mapping
# ==============================================================================
target_s100_genes = ['S100A2', 'S100A3', 'S100A4', 'S100A5', 'S100A6', 'SERPINE1']

# Dataset 8: use biological donor mapping (9 donors, not 21 samples)
res_d8_rna = run_rna_pseudobulk(
    adata_obj=ds8,
    dataset_name="Dataset 8",
    cluster_col="leiden_res_2.0",
    act_clusters=["2"],
    homeo_clusters=["0", "1"],
    donor_col="donor",
    target_genes=target_s100_genes
)

# Dataset 9: each sample is a unique donor
res_d9_rna = run_rna_pseudobulk(
    adata_obj=ds9,
    dataset_name="Dataset 9",
    cluster_col="leiden_res_2.0",
    act_clusters=["6", "8"],
    homeo_clusters=["2", "4", "5"],
    donor_col="sample",
    target_genes=target_s100_genes
)

# Consolidate and apply Benjamini-Hochberg FDR correction
valid_dfs = [df for df in [res_d8_rna, res_d9_rna] if df is not None and not df.empty]
if valid_dfs:
    df_rna_final = pd.concat(valid_dfs, ignore_index=True)
    df_rna_final['p_value'] = df_rna_final['p_value'].fillna(1.0)
    _, p_adjs, _, _ = multipletests(df_rna_final['p_value'], method='fdr_bh')
    df_rna_final['p_adj'] = p_adjs
else:
    df_rna_final = pd.DataFrame()
    print("Error: Both processed DataFrames returned empty.")

df_rna_final

## **Paired visualization of S100A4 and SERPINE1 expression**

Single-cell expression distribution (violin + cell dots) overlaid with donor-level pseudobulk means (colored circles + connecting lines). Each pair of points connected by a line represents one biological donor. The paired t-test p-value is computed on the pseudobulk means across donors.

In [ ]:
# ============================================================================
# Paired violin + sand-dots + donor-line visualization for S100A4 & SERPINE1
# ============================================================================

def plot_paired_expression_combined(
    adata_8, adata_9,
    target_genes,
    act_clusters_8, homeo_clusters_8,
    act_clusters_9, homeo_clusters_9,
    donor_col_8="donor", donor_col_9="sample",
    n_rows=None
):
    """
    Generates paired violin + strip + donor-line plots for multiple genes.
    Each gene gets one row (Dataset 8 left, Dataset 9 right).
    P-values are computed from a paired t-test on donor-level pseudobulk means.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import pandas as pd
    from scipy.stats import ttest_rel

    # ==============================================================
    # COLOR AND SIZE SETTINGS
    # ==============================================================
    _C_VIOLIN_HOMEO = "#A020F0"
    _C_VIOLIN_GPNMB = "#FF7F00"
    VIOLIN_ALPHA    = 0.4

    _C_CELL_DOTS    = "#000000"
    CELL_DOTS_SIZE  = 2
    CELL_DOTS_ALPHA = 1.0

    _C_DONOR_HOMEO  = "#A020F0"
    _C_DONOR_GPNMB  = "#FF7F00"
    DONOR_DOTS_SIZE = 55
    DONOR_ALPHA     = 0.7
    DONOR_EDGE_COLOR = "black"

    _C_CONNECTED_LINE = "#95a5a6"
    CONNECTED_LINE_W  = 1.0
    CONNECTED_LINE_A  = 0.6

    palette_violin = {"Homeostatic": _C_VIOLIN_HOMEO, "GPNMB+": _C_VIOLIN_GPNMB}

    experimental_setup = [
        ("Dataset 8", adata_8, act_clusters_8, homeo_clusters_8, donor_col_8),
        ("Dataset 9", adata_9, act_clusters_9, homeo_clusters_9, donor_col_9),
    ]

    n_genes = len(target_genes)
    n_cols = 2  # Dataset 8 | Dataset 9

    # Halved height: each gene row ~4 units, total = n_genes * 4
    row_height = 4
    fig, axs = plt.subplots(n_genes, n_cols, figsize=(13, n_genes * row_height), dpi=300, squeeze=False)

    for gene_idx, target_gene in enumerate(target_genes):
        for ds_idx, (dataset_name, adata_obj, act_clusters, homeo_clusters, donor_col) in enumerate(experimental_setup):
            ax = axs[gene_idx, ds_idx]

            # 1. Extract single-cell expression
            symbol_to_var = dict(zip(adata_obj.var["gene_symbols"], adata_obj.var_names))
            if target_gene not in symbol_to_var:
                ax.text(0.5, 0.5, f"{target_gene} not found", transform=ax.transAxes, ha="center")
                continue
            feature_id = symbol_to_var[target_gene]

            all_clusters = act_clusters + homeo_clusters
            cell_mask = adata_obj.obs["leiden_res_2.0"].isin(all_clusters)

            if hasattr(adata_obj[:, feature_id].X, "toarray"):
                exp_vec = adata_obj[cell_mask, feature_id].X.toarray().flatten()
            else:
                exp_vec = np.asarray(adata_obj[cell_mask, feature_id].X).flatten()

            df_cells = pd.DataFrame({
                "Expression": exp_vec,
                "donor_id": adata_obj.obs.loc[cell_mask, donor_col].astype(str),
                "state": adata_obj.obs.loc[cell_mask, "leiden_res_2.0"].apply(
                    lambda x: "GPNMB+" if x in act_clusters else "Homeostatic"
                )
            })

            # 2. Violin plot (background density)
            sns.violinplot(
                data=df_cells, x="state", y="Expression",
                order=["Homeostatic", "GPNMB+"],
                hue="state", palette=palette_violin, legend=False,
                inner=None, linewidth=1.2, alpha=VIOLIN_ALPHA,
                width=0.6, cut=0, ax=ax, zorder=1
            )

            # 3. Cell-level strip plot (sand dots)
            df_cells_nonzero = df_cells[df_cells["Expression"] > 0]
            if len(df_cells_nonzero) > 0:
                sns.stripplot(
                    data=df_cells_nonzero, x="state", y="Expression",
                    order=["Homeostatic", "GPNMB+"],
                    color=_C_CELL_DOTS, dodge=False, size=CELL_DOTS_SIZE,
                    alpha=CELL_DOTS_ALPHA, jitter=0.25,
                    edgecolor=None, linewidth=0, ax=ax, zorder=2
                )

            # 4. Donor-level pseudobulk (paired)
            df_pb = df_cells.groupby(["donor_id", "state"])["Expression"].mean().reset_index()
            paired_donors = df_pb.groupby("donor_id")["state"].nunique()
            paired_donors = paired_donors[paired_donors == 2].index.tolist()
            df_paired = df_pb[df_pb["donor_id"].isin(paired_donors)]

            # Compute paired t-test p-value
            p_val_text = ""
            if len(paired_donors) >= 2:
                act_vals = df_paired[df_paired["state"] == "GPNMB+"].sort_values("donor_id")["Expression"].values
                homeo_vals = df_paired[df_paired["state"] == "Homeostatic"].sort_values("donor_id")["Expression"].values
                if len(act_vals) == len(homeo_vals) and len(act_vals) >= 2:
                    _, p_val = ttest_rel(act_vals, homeo_vals)
                    p_val_text = f"p = {p_val:.2e}"

            # Fixed jitter for donor dots
            np.random.seed(42)
            jitters = {d: np.random.uniform(-0.12, 0.12) for d in paired_donors}

            # 5. Draw paired lines and donor circles
            for donor in paired_donors:
                df_d = df_paired[df_paired["donor_id"] == donor]
                val_homeo = df_d[df_d["state"] == "Homeostatic"]["Expression"].values.item()
                val_gpnmb = df_d[df_d["state"] == "GPNMB+"]["Expression"].values.item()

                j = jitters[donor]
                x_coords = [0 + j, 1 + j]
                y_coords = [val_homeo, val_gpnmb]

                ax.plot(x_coords, y_coords,
                        color=_C_CONNECTED_LINE, alpha=CONNECTED_LINE_A,
                        linewidth=CONNECTED_LINE_W, zorder=3)

                ax.scatter(x_coords[0], y_coords[0],
                           color=_C_DONOR_HOMEO, edgecolor=DONOR_EDGE_COLOR,
                           s=DONOR_DOTS_SIZE, linewidth=0.8, alpha=DONOR_ALPHA, zorder=4)
                ax.scatter(x_coords[1], y_coords[1],
                           color=_C_DONOR_GPNMB, edgecolor=DONOR_EDGE_COLOR,
                           s=DONOR_DOTS_SIZE, linewidth=0.8, alpha=DONOR_ALPHA, zorder=4)

            # Axis styling
            ax.set_xlim(-0.4, 1.4)
            ax.set_xlabel("")
            y_max = df_cells["Expression"].max()
            ax.set_ylim(-0.02, y_max * 1.05)

            # Gene name on the left for first column only
            if ds_idx == 0:
                ax.set_ylabel(f"{target_gene} Expression", fontsize=12, fontstyle="italic")
            else:
                ax.set_ylabel("")

            # Title with dataset name, donor count and p-value
            title_str = f"{dataset_name}: {len(paired_donors)} paired donors"
            if p_val_text:
                title_str += f"\n{p_val_text}"
            ax.set_title(title_str, fontsize=11, fontweight="bold")
            ax.tick_params(labelsize=10)

    sns.despine()
    plt.tight_layout()
    plt.show()


# ==============================================================================
# EXECUTION: plot S100A4 and SERPINE1
# ==============================================================================
plot_paired_expression_combined(
    adata_8=ds8,
    adata_9=ds9,
    target_genes=["S100A4", "SERPINE1"],
    act_clusters_8=["2"],
    homeo_clusters_8=["0", "1"],
    act_clusters_9=["6", "8"],
    homeo_clusters_9=["2", "4", "5"],
    donor_col_8="donor",   # 9 biological donors
    donor_col_9="sample",  # each sample = unique donor
)